# ETL Notebook
---
Running this notebook from end-to-end will reconstruct all tables from the raw sources

In [312]:
%env DATABASE_URL=postgresql://healthgps:healthgps@db/healthgps

env: DATABASE_URL=postgresql://healthgps:healthgps@db/healthgps


In [ ]:
"""Run this to establish the connection with the database"""
# load the SQL extension
%load_ext sql

# create the connection string
import os
user = os.getenv('postgres_user')
password = os.getenv('postgres_password')
db = os.getenv('postgres_db')
db_url = f"postgresql://{user}:{password}@db/{db}"

# establish the connection
from sqlalchemy import create_engine

# engine = create_engine(db_url)

# set %sql magic commands to use the connection
# %sql engine

In [318]:
%%sql
-- Create states table from raw table
DROP TABLE states CASCADE;

CREATE TABLE
  IF NOT EXISTS states (
    id VARCHAR(2) PRIMARY KEY,
    fp VARCHAR(2),
    name VARCHAR(255),
    code VARCHAR(2),
    division_id VARCHAR(1),
    region_id VARCHAR(1),
    geometry GEOMETRY (MULTIPOLYGON, 4326)
  );

CREATE INDEX states_geom_idx ON states USING GIST (geometry);

INSERT INTO
  states (
    id,
    fp,
    name,
    code,
    division_id,
    region_id,
    geometry
  )
SELECT
  "GEOID" AS id,
  "STATEFP" AS fp,
  UPPER("NAME") AS name,
  "STUSPS" AS code,
  "DIVISION" AS division_id,
  "REGION" AS region_id,
  geometry
FROM
  "census_2024_state_raw";

 * postgresql://healthgps:***@db/healthgps
Done.
Done.
Done.
56 rows affected.


[]

In [320]:
%%sql
-- Create counties table from raw table
DROP TABLE counties;

CREATE TABLE
  counties (
    id VARCHAR(5) PRIMARY KEY,
    fp VARCHAR(3),
    name VARCHAR(255),
    state_name VARCHAR(255),
    state_code VARCHAR(2),
    state_id VARCHAR(2),
    division_id VARCHAR(1),
    region_id VARCHAR(1),
    geometry GEOMETRY (MULTIPOLYGON, 4326)
  );

CREATE INDEX counties_geom_idx ON counties USING GIST (geometry);

INSERT INTO
  counties (
    id,
    fp,
    name,
    state_name,
    state_code,
    state_id,
    division_id,
    region_id,
    geometry
  )
SELECT
  "GEOID" AS id,
  "COUNTYFP" AS fp,
  UPPER("NAME") AS name,
  states.name AS state_name,
  states.code AS state_code,
  states.id AS state_id,
  states.division_id,
  states.region_id,
  counties.geometry AS geometry
FROM
  "census_2024_county_raw" counties
  LEFT JOIN states ON states.fp = counties."STATEFP";

 * postgresql://healthgps:***@db/healthgps
Done.
Done.
Done.
3235 rows affected.


[]

In [321]:
%%sql
-- Create tracts table from raw table
DROP TABLE tracts;

CREATE TABLE
  tracts (
    id VARCHAR(11) PRIMARY KEY,
    fp VARCHAR(6),
    name VARCHAR(255),
    county_name VARCHAR(255),
    state_name VARCHAR(255),
    state_code VARCHAR(2),
    county_id VARCHAR(5),
    state_id VARCHAR(2),
    division_id VARCHAR(1),
    region_id VARCHAR(1),
    geometry GEOMETRY (MULTIPOLYGON, 4326)
  );

CREATE INDEX tracts_geom_idx ON tracts USING GIST (geometry);

INSERT INTO
  tracts (
    id,
    fp,
    name,
    county_name,
    state_name,
    state_code,
    county_id,
    state_id,
    division_id,
    region_id,
    geometry
  )
SELECT
  "GEOID" AS id,
  "TRACTCE" AS fp,
  UPPER("NAME") AS name,
  counties.name AS county_name,
  counties.state_name AS state_name,
  counties.state_code AS state_code,
  counties.id AS county_id,
  counties.state_id AS state_id,
  counties.division_id AS division_id,
  counties.region_id AS region_id,
  tracts.geometry AS geometry
FROM
  "census_2024_tract_raw" tracts
  LEFT JOIN counties ON counties.id = tracts."STATEFP" || tracts."COUNTYFP";

 * postgresql://healthgps:***@db/healthgps
Done.
Done.
Done.
85529 rows affected.


[]

In [322]:
%%sql
-- Create block groups table from raw table
DROP TABLE blkgrps;

CREATE TABLE
  blkgrps (
    id VARCHAR(12) PRIMARY KEY,
    fp VARCHAR(1),
    name VARCHAR(255),
    tract_name VARCHAR(255),
    county_name VARCHAR(255),
    state_name VARCHAR(255),
    state_code VARCHAR(2),
    tract_id VARCHAR(11),
    county_id VARCHAR(5),
    state_id VARCHAR(2),
    division_id VARCHAR(1),
    region_id VARCHAR(1),
    geometry GEOMETRY (MULTIPOLYGON, 4326)
  );

CREATE INDEX blkgrps_geom_idx ON blkgrps USING GIST (geometry);

INSERT INTO
  blkgrps (
    id,
    fp,
    name,
    tract_name,
    county_name,
    state_name,
    state_code,
    tract_id,
    county_id,
    state_id,
    division_id,
    region_id,
    geometry
  )
SELECT
  "GEOID" AS id,
  "BLKGRPCE" AS fp,
  UPPER("NAMELSAD") AS name,
  tracts.name AS tract_name,
  tracts.county_name AS county_name,
  tracts.state_name AS state_name,
  tracts.state_code AS state_code,
  tracts.id AS tract_id,
  tracts.county_id AS county_id,
  tracts.state_id AS state_id,
  tracts.division_id AS division_id,
  tracts.region_id AS region_id,
  blocks.geometry AS geometry
FROM
  "census_2024_blkgrp_raw" blocks
  LEFT JOIN tracts ON tracts.id = blocks."STATEFP" || blocks."COUNTYFP" || blocks."TRACTCE";

 * postgresql://healthgps:***@db/healthgps
Done.
Done.
Done.
242748 rows affected.


[]

In [294]:
%%sql
-- Add unique identifiers for each pantry row for tracing
ALTER TABLE efos_raw
ADD COLUMN traceid UUID DEFAULT (gen_random_uuid ());

 * postgresql://healthgps:***@db/healthgps
Done.


[]

In [297]:
%%sql
-- Create foodbanks table
DROP TABLE efos CASCADE;

CREATE TABLE
  efos (
    id UUID PRIMARY KEY,
    created_time TIMESTAMP WITH TIME ZONE,
    modified_time TIMESTAMP WITH TIME ZONE,
    uploaded_time TIMESTAMP WITH TIME ZONE,
    organization_id VARCHAR(24),
    name VARCHAR(2040),
    source_url VARCHAR(2040),
    reference_id VARCHAR(255),
    type TEXT ARRAY,
    latitude FLOAT,
    longitude FLOAT,
    address_full TEXT,
    address_number VARCHAR(255),
    street_name VARCHAR(255),
    street_name_post_type VARCHAR(255),
    occupancy_type VARCHAR(255),
    occupancy_identifier VARCHAR(255),
    city_name VARCHAR(255),
    state_code_orig VARCHAR(255),
    country_code VARCHAR(255),
    zip_code VARCHAR(255),
    phone_number VARCHAR(255),
    metadata JSONB,
    duplicate BOOLEAN,
    blkgrp_name VARCHAR(255),
    tract_name VARCHAR(255),
    county_name VARCHAR(255),
    state_name VARCHAR(255),
    state_code VARCHAR(2),
    blkgrp_id VARCHAR(12),
    tract_id VARCHAR(11),
    county_id VARCHAR(5),
    state_id VARCHAR(2),
    division_id VARCHAR(1),
    region_id VARCHAR(1),
    geometry GEOMETRY (POINT, 4326)
  );

CREATE INDEX efos_geom_idx ON efos USING GIST (geometry);

INSERT INTO
  efos (
    id,
    created_time,
    modified_time,
    uploaded_time,
    organization_id,
    name,
    source_url,
    reference_id,
    type,
    latitude,
    longitude,
    address_full,
    address_number,
    street_name,
    street_name_post_type,
    occupancy_type,
    occupancy_identifier,
    city_name,
    state_code_orig,
    country_code,
    zip_code,
    phone_number,
    metadata,
    duplicate,
    blkgrp_name,
    tract_name,
    county_name,
    state_name,
    state_code,
    blkgrp_id,
    tract_id,
    county_id,
    state_id,
    division_id,
    region_id,
    geometry
  )
SELECT
    DISTINCT ON (name, organization_id, type, address_full, address_number, street_name, street_name_post_type, occupancy_type, occupancy_identifier, city_name, state_code_orig, country_code, zip_code, phone_number, geometry)
    traceid as id,
    efo."createdTime"::timestamp with time zone as created_time,
    efo."modifiedTime"::timestamp with time zone as modified_time,
    efo."created_at" as uploaded_time,
    efo."organizationId" as organization_id,
    efo."name" as name,
    efo."sourceUrl" as source_url,
    efo."referenceId" as reference_id,
    string_to_array(replace(replace(replace(replace(efo."type",'[',''),']',''),'''''', ''),', ',','),',') as type,
    efo."latitude" as latitude,
    efo."longitude" as longitude,
    efo."addressFull" as address_full,
    efo."addressNumber" as address_number,
    efo."streetName" as street_name,
    efo."streetNamePostType" as street_name_post_type,
    efo."occupancyType" as occupancy_type,
    efo."occupancyIdentifier" as occupancy_identifier,
    efo."cityName" as city_name,
    efo."stateCode" as state_code_orig,
    efo."countryCode" as country_code,
    efo."zipCode" as zip_code,
    efo."phoneNumber" as phone_number,
    to_jsonb(metadata) as metadata,
    False as duplicate,
    blkgrps.name AS blkgrp_name,
    blkgrps.tract_name AS tract_name,
    blkgrps.county_name AS county_name,
    blkgrps.state_name AS state_name,
    blkgrps.state_code AS state_code,
    blkgrps.id AS blkgrp_id,
    blkgrps.tract_id AS tract_id,
    blkgrps.county_id AS county_id,
    blkgrps.state_id AS state_id,
    blkgrps.division_id AS division_id,
    blkgrps.region_id AS region_id,
    efo.geometry
FROM blkgrps
INNER JOIN efos_raw efo ON st_contains (blkgrps.geometry, efo.geometry)

 * postgresql://healthgps:***@db/healthgps
Done.
Done.
Done.
122161 rows affected.


[]

In [304]:
%%sql
-- Create clusters from duplicate records

DROP TABLE efos_clusters CASCADE;

CREATE TABLE
  efos_clusters (
    id UUID PRIMARY KEY,
    created_time TIMESTAMP WITH TIME ZONE,
    modified_time TIMESTAMP WITH TIME ZONE,
    uploaded_time TIMESTAMP WITH TIME ZONE,
    organization_id VARCHAR(24),
    name VARCHAR(2040),
    source_url VARCHAR(2040),
    reference_id VARCHAR(2040),
    type TEXT ARRAY,
    latitude FLOAT,
    longitude FLOAT,
    address_full TEXT,
    address_number VARCHAR(255),
    street_name VARCHAR(255),
    street_name_post_type VARCHAR(255),
    occupancy_type VARCHAR(255),
    occupancy_identifier VARCHAR(255),
    city_name VARCHAR(255),
    state_code_orig VARCHAR(255),
    country_code VARCHAR(255),
    zip_code VARCHAR(255),
    phone_number VARCHAR(255),
    metadata JSONB,
    duplicate BOOLEAN,
    blkgrp_name VARCHAR(255),
    tract_name VARCHAR(255),
    county_name VARCHAR(255),
    state_name VARCHAR(255),
    state_code VARCHAR(2),
    blkgrp_id VARCHAR(12),
    tract_id VARCHAR(11),
    county_id VARCHAR(5),
    state_id VARCHAR(2),
    division_id VARCHAR(1),
    region_id VARCHAR(1),
    geometry GEOMETRY (POINT, 4326)
  );

CREATE INDEX efos_clusters_geom_idx ON efos_clusters USING GIST (geometry);

insert into efos_clusters (
    id,
    created_time,
    modified_time,
    uploaded_time,
    organization_id,
    name,
    source_url,
    reference_id,
    type,
    latitude,
    longitude,
    address_full,
    address_number,
    street_name,
    street_name_post_type,
    occupancy_type,
    occupancy_identifier,
    city_name,
    state_code_orig,
    country_code,
    zip_code,
    phone_number,
    metadata,
    duplicate,
    blkgrp_name,
    tract_name,
    county_name,
    state_name,
    state_code,
    blkgrp_id,
    tract_id,
    county_id,
    state_id,
    division_id,
    region_id,
    geometry
)
with knn as (
    select
        a.id as a_id,
        b.id as b_id,
        a.name as a_name,
        b.name as b_name,
        a.type as a_type,
        b.type as b_type,
        a.address_full as a_address,
        b.address_full as b_address,
        a.street_name as a_street_name,
        b.street_name as b_street_name,
        a.street_name_post_type as a_street_name_post_type,
        b.street_name_post_type as b_street_name_post_type,
        a.occupancy_type as a_occupancy_type,
        b.occupancy_type as b_occupancy_type,
        a.occupancy_identifier as a_occupancy_identifier,
        b.occupancy_identifier as b_occupancy_identifier,
        a.address_number as a_address_number,
        b.address_number as b_address_number,
        a.phone_number as a_phone_number,
        b.phone_number as b_phone_number,
        a.geometry as a_geometry,
        b.geometry as b_geometry,
        a.blkgrp_name as a_blkgrp_name,
        b.blkgrp_name as b_blkgrp_name,
        a.tract_name as a_tract_name,
        b.tract_name as b_tract_name,
        a.county_name as a_county_name,
        b.county_name as b_county_name,
        a.state_code as a_state_code,
        b.state_code as b_state_code,
        st_distance(a.geometry::geography, b.geometry::geography) as dist
    from efos a
    cross join lateral (
        select
            id,
            name,
            type,
            address_full,
            street_name,
            street_name_post_type,
            occupancy_type,
            occupancy_identifier,
            address_number,
            phone_number,
            blkgrp_name,
            tract_name,
            county_name,
            state_code,
            geometry
        from efos f
        where f.id <> a.id
        order by f.geometry <-> a.geometry
        limit 10 -- "k"
    ) b
),
topk as (
    select
        *
    from knn
    where
        (
            dist < 150
            and a_phone_number = b_phone_number
        )
        or
        (
            dist < 150
            and a_street_name = b_street_name
            and a_address_number = b_address_number
        )
        or
        (
            dist < 50
            and a_address_number = b_address_number
        )
        or
        (
            dist < 50
            and position(a_name IN b_name) > 0
        )
        or
        (
            dist < 50
            and similarity(a_name, b_name) > 0.8
        )
),
clusters as (
    select
        distinct cluster_id
    FROM (
         SELECT
            a_id as primary_node,
            ARRAY(SELECT element FROM unnest(array_cat(array[a_id], array_agg(b_id))) AS element ORDER BY element) as cluster_id
        FROM
            topk
        GROUP BY a_id
    ) sub
),
superset_clusters AS (
    SELECT
        a.cluster_id AS cluster_id,
        b.cluster_id AS parent_cluster_id
    FROM clusters a
    CROSS JOIN LATERAL (
        SELECT
            cluster_id
        FROM clusters b
        WHERE a.cluster_id <@ b.cluster_id
        ORDER BY cardinality(b.cluster_id) DESC
        LIMIT 1
    ) b
),
deduped_clusters AS (
    SELECT
        distinct parent_cluster_id AS cluster_id
    FROM superset_clusters
)
SELECT
    gen_random_uuid() as id,
    now() as created_time,
    now() as modified_time,
    now() as uploaded_time,
    any_value(n.organization_id) filter (where n.id = c.cluster_id[1])as organization_id,
    any_value(n.name) filter (where n.id = c.cluster_id[1])as name,
    'cluster' as source_url,
    c.cluster_id as reference_id,
    array(select distinct unnest from unnest(array_agg(elem))) as type,
    ST_Y(ST_Centroid(ST_Transform(st_centroid(st_union(n.geometry)), 4326))) as latitude,
    ST_X(ST_Centroid(ST_Transform(st_centroid(st_union(n.geometry)), 4326))) as longitude,
    any_value(n.address_full) filter (where n.id = c.cluster_id[1])as address_full,
    any_value(n.address_number) filter (where n.id = c.cluster_id[1])as address_number,
    any_value(n.street_name) filter (where n.id = c.cluster_id[1])as street_name,
    any_value(n.street_name_post_type) filter (where n.id = c.cluster_id[1])as street_name_post_type,
    any_value(n.occupancy_type) filter (where n.id = c.cluster_id[1])as occupancy_type,
    any_value(n.occupancy_identifier) filter (where n.id = c.cluster_id[1])as occupancy_identifier,
    any_value(n.city_name) filter (where n.id = c.cluster_id[1])as city_name,
    any_value(n.state_code_orig) filter (where n.id = c.cluster_id[1])as state_code_orig,
    any_value(n.country_code) filter (where n.id = c.cluster_id[1])as country_code,
    any_value(n.zip_code) filter (where n.id = c.cluster_id[1])as zip_code,
    string_agg(distinct n.phone_number, ',') as phone_number,
    jsonb_build_object('count', count(*), 'metadata', jsonb_agg(n.metadata)) as metadata,
    False as duplicate,
    any_value(n.blkgrp_name) filter (where n.id = c.cluster_id[1])as blkgrp_name,
    any_value(n.tract_name) filter (where n.id = c.cluster_id[1])as tract_name,
    any_value(n.county_name) filter (where n.id = c.cluster_id[1])as county_name,
    any_value(n.state_name) filter (where n.id = c.cluster_id[1])as state_name,
    any_value(n.state_code) filter (where n.id = c.cluster_id[1])as state_code,
    any_value(n.blkgrp_id) filter (where n.id = c.cluster_id[1])as blkgrp_id,
    any_value(n.tract_id) filter (where n.id = c.cluster_id[1])as tract_id,
    any_value(n.county_id) filter (where n.id = c.cluster_id[1])as county_id,
    any_value(n.state_id) filter (where n.id = c.cluster_id[1])as state_id,
    any_value(n.division_id) filter (where n.id = c.cluster_id[1])as division_id,
    any_value(n.region_id) filter (where n.id = c.cluster_id[1])as region_id,
    st_centroid(st_union(n.geometry)) as geometry
from deduped_clusters c
join efos n
    on n.id = any(c.cluster_id)
cross join unnest(n.type) as elem
group by c.cluster_id

 * postgresql://healthgps:***@db/healthgps
Done.
Done.
Done.
30047 rows affected.


[]

In [127]:
%%sql
-- Set all those individual points as duplicates
-- may have to run in sql terminal

UPDATE efos fo
SET duplicate = TRUE
FROM (
    SELECT
        DISTINCT uu.uuid::uuid
    FROM efos_clusters
    CROSS JOIN LATERAL unnest(string_to_array(replace(replace(reference_id,'{',''),'}',''),',')) AS uu(uuid)
) dup
where fo.id = dup.uuid;

 * postgresql://healthgps:***@db/healthgps
(psycopg2.errors.SyntaxError) syntax error at or near "AS"
LINE 7: ...rray(replace(replace(reference_id,',),',''),',')) AS uu(uuid...
                                                             ^

[SQL: UPDATE efos fo
SET duplicate = TRUE
FROM (
    SELECT
        DISTINCT uu.uuid::uuid
    FROM efos_clusters
    CROSS JOIN LATERAL unnest(string_to_array(replace(replace(reference_id,',),',''),',')) AS uu(uuid)
) dup
where fo.id = dup.uuid;]
(Background on this error at: https://sqlalche.me/e/20/f405)


In [307]:
%%sql
-- Filter out duplicates through a view
DROP VIEW efos_deduped;

CREATE VIEW
    efos_deduped AS
SELECT
  *
FROM
  efos
WHERE
  duplicate = False

UNION ALL

SELECT
    *
FROM
    efos_clusters;

 * postgresql://healthgps:***@db/healthgps
Done.


[]

In [133]:
%%sql
-- Add unique identifiers for each pantry row for tracing
ALTER TABLE snap_retailers_raw
ADD COLUMN traceid UUID DEFAULT (gen_random_uuid ());

 * postgresql://healthgps:***@db/healthgps
Done.


[]

In [308]:
%%sql
-- SNAP retailers
DROP TABLE snap_retailers;

CREATE TABLE
  snap_retailers (
    id UUID PRIMARY KEY,
    source VARCHAR(255),
    alt_id VARCHAR(255),
    name VARCHAR(255),
    address VARCHAR(255),
    type VARCHAR(255),
    blkgrp_name VARCHAR(255),
    tract_name VARCHAR(255),
    county_name VARCHAR(255),
    state_name VARCHAR(255),
    state_code VARCHAR(2),
    blkgrp_id VARCHAR(12),
    tract_id VARCHAR(11),
    county_id VARCHAR(5),
    state_id VARCHAR(2),
    division_id VARCHAR(1),
    region_id VARCHAR(1),
    geometry GEOMETRY (POINT, 4326)
  );

CREATE INDEX snap_retailers_geom_idx ON snap_retailers USING GIST (geometry);

INSERT INTO
  snap_retailers (
    id,
    source,
    alt_id,
    name,
    address,
    type,
    blkgrp_name,
    tract_name,
    county_name,
    state_name,
    state_code,
    blkgrp_id,
    tract_id,
    county_id,
    state_id,
    division_id,
    region_id,
    geometry
  )
SELECT
  traceid AS id,
  'usda' AS source,
  snap."Record_ID" AS alt_id,
  UPPER(snap."Store_Name") AS name,
  UPPER(
    concat_ws (
      ', ',
      "Store_Street_Address",
      "City",
      "State",
      "Zip_Code"
    )
  ) AS address,
  UPPER(snap."Store_Type") AS type,
  blkgrps.name AS blkgrp_name,
  blkgrps.tract_name AS tract_name,
  blkgrps.county_name AS county_name,
  blkgrps.state_name AS state_name,
  blkgrps.state_code AS state_code,
  blkgrps.id AS blkgrp_id,
  blkgrps.tract_id AS tract_id,
  blkgrps.county_id AS county_id,
  blkgrps.state_id AS state_id,
  blkgrps.division_id AS division_id,
  blkgrps.region_id AS region_id,
  snap.geometry
FROM
  blkgrps
  INNER JOIN snap_retailers_raw snap ON st_contains (blkgrps.geometry, snap.geometry)

 * postgresql://healthgps:***@db/healthgps
Done.
Done.
Done.
260069 rows affected.


[]

In [326]:
%%sql
-- Create analytical county table
DROP TABLE counties_metrics;

CREATE TABLE counties_metrics (
    id VARCHAR(5) PRIMARY KEY,
    ct_population INTEGER,
    ct_households INTEGER,
    ct_households_any_food_assistance INTEGER,
    ct_households_snap_food_assistance INTEGER,
    ct_households_snap_food_assistance_disability INTEGER,
    ct_housingunits INTEGER,
    ct_housingunits_owner_occupied INTEGER,
    ct_housingunits_owner_occupied_no_vehicle INTEGER,
    ct_housingunits_renter_occupied INTEGER,
    ct_housingunits_renter_occupied_no_vehicle INTEGER,
    ct_poverty_determined_population INTEGER,
    ct_poverty_income_ratio_lt_50pct INTEGER,
    ct_poverty_income_ratio_lte_99pct INTEGER,
    ct_poverty_income_ratio_lte_124pct INTEGER,
    ct_poverty_income_ratio_lte_149pct INTEGER,
    ct_poverty_income_ratio_lte_184pct INTEGER,
    ct_poverty_income_ratio_lte_199pct INTEGER,
    ct_poverty_income_ratio_gt_200pct INTEGER,
    ct_income_below_poverty INTEGER,
    ct_income_below_poverty_lt_6yo INTEGER,
    ct_income_below_poverty_lte_11yo INTEGER,
    ct_income_below_poverty_lte_17yo INTEGER,
    ct_snap_retailers INTEGER,
    ct_food_outposts INTEGER,
    fp VARCHAR(3),
    name VARCHAR(255),
    state_name VARCHAR(255),
    state_code VARCHAR(2),
    state_id VARCHAR(2),
    division_id VARCHAR(1),
    region_id VARCHAR(1)
);

INSERT INTO counties_metrics (
    id,
    ct_population,
    ct_households,
    ct_households_any_food_assistance,
    ct_households_snap_food_assistance,
    ct_households_snap_food_assistance_disability,
    ct_housingunits,
    ct_housingunits_owner_occupied,
    ct_housingunits_owner_occupied_no_vehicle,
    ct_housingunits_renter_occupied,
    ct_housingunits_renter_occupied_no_vehicle,
    ct_poverty_determined_population,
    ct_poverty_income_ratio_lt_50pct,
    ct_poverty_income_ratio_lte_99pct,
    ct_poverty_income_ratio_lte_124pct,
    ct_poverty_income_ratio_lte_149pct,
    ct_poverty_income_ratio_lte_184pct,
    ct_poverty_income_ratio_lte_199pct,
    ct_poverty_income_ratio_gt_200pct,
    ct_income_below_poverty,
    ct_income_below_poverty_lt_6yo,
    ct_income_below_poverty_lte_11yo,
    ct_income_below_poverty_lte_17yo,
    ct_snap_retailers,
    ct_food_outposts,
    fp,
    name,
    state_name,
    state_code,
    state_id,
    division_id,
    region_id
)

WITH snap_agg AS (
  SELECT
    cty.id,
    COUNT(DISTINCT snap.id) AS ct_snap_retailers
  FROM counties cty
  LEFT JOIN snap_retailers snap
    ON cty.geometry && snap.geometry
   AND ST_Contains(cty.geometry, snap.geometry)
  GROUP BY cty.id
),
outpost_agg AS (
  SELECT
    cty.id,
    COUNT(DISTINCT o.id) AS ct_food_outposts
  FROM counties cty
  LEFT JOIN efos_deduped o
    ON cty.geometry && o.geometry
   AND ST_Contains(cty.geometry, o.geometry)
  GROUP BY cty.id
),
agg AS (
  SELECT cty.id,
         COALESCE(s.ct_snap_retailers, 0) AS ct_snap_retailers,
         COALESCE(o.ct_food_outposts, 0) AS ct_food_outposts
  FROM counties cty
  LEFT JOIN snap_agg s USING (id)
  LEFT JOIN outpost_agg o USING (id)
)
SELECT
    cty.id,
    survey."ASN1E001"::integer as ct_population,
    survey."ASRAE001"::integer as ct_households,
    survey."ASRAE002"::integer as ct_households_any_food_assistance,
    survey."ASSKE002"::integer as ct_households_snap_food_assistance,
    survey."ASSKE003"::integer as ct_households_snap_food_assistance_disability,
    survey."ASUTE002"::integer as ct_housingunits,
    survey."ASUTE002"::integer as ct_housingunits_owner_occupied,
    survey."ASUTE003"::integer as ct_housingunits_owner_occupied_no_vehicle,
    survey."ASUTE009"::integer as ct_housingunits_renter_occupied,
    survey."ASUTE010"::integer as ct_housingunits_renter_occupied_no_vehicle,
    survey."ASQIE001"::integer as ct_poverty_determined_population,
    survey."ASQIE002"::integer as ct_poverty_income_ratio_lt_50pct,
    survey."ASQIE003"::integer as ct_poverty_income_ratio_lte_99pct,
    survey."ASQIE004"::integer as ct_poverty_income_ratio_lte_124pct,
    survey."ASQIE005"::integer as ct_poverty_income_ratio_lte_149pct,
    survey."ASQIE006"::integer as ct_poverty_income_ratio_lte_184pct,
    survey."ASQIE007"::integer as ct_poverty_income_ratio_lte_199pct,
    survey."ASQIE008"::integer as ct_poverty_income_ratio_gt_200pct,
    survey."AS7TE002"::integer as ct_income_below_poverty,
    survey."AS7TE003"::integer as ct_income_below_poverty_lt_6yo,
    survey."AS7TE004"::integer as ct_income_below_poverty_lte_11yo,
    survey."AS7TE005"::integer as ct_income_below_poverty_lte_17yo,
    agg.ct_snap_retailers as ct_snap_retailers,
    agg.ct_food_outposts as ct_food_outposts,
    cty.fp,
    cty.name,
    cty.state_name,
    cty.state_code,
    cty.state_id,
    cty.division_id,
    cty.region_id
from counties cty
INNER JOIN "acs_2023_county_raw" survey
    ON cty.id = survey."TL_GEO_ID"
LEFT JOIN agg
    ON cty.id = agg.id

 * postgresql://healthgps:***@db/healthgps
Done.
Done.


KeyboardInterrupt: 

In [324]:
%%sql
-- Create analytical state table

DROP TABLE state_metrics;

CREATE TABLE state_metrics (
    id VARCHAR(2),
    ct_population INTEGER,
    ct_households INTEGER,
    ct_households_any_food_assistance INTEGER,
    ct_households_snap_food_assistance INTEGER,
    ct_households_snap_food_assistance_disability INTEGER,
    ct_housingunits INTEGER,
    ct_housingunits_owner_occupied INTEGER,
    ct_housingunits_owner_occupied_no_vehicle INTEGER,
    ct_housingunits_renter_occupied INTEGER,
    ct_housingunits_renter_occupied_no_vehicle INTEGER,
    ct_snap_retailers INTEGER,
    ct_food_outposts INTEGER,
    name VARCHAR(255),
    code VARCHAR(2),
    division_id VARCHAR(1),
    region_id VARCHAR(1)
);


INSERT INTO state_metrics (
    id,
    ct_population,
    ct_households,
    ct_households_any_food_assistance,
    ct_households_snap_food_assistance,
    ct_households_snap_food_assistance_disability,
    ct_housingunits,
    ct_housingunits_owner_occupied,
    ct_housingunits_owner_occupied_no_vehicle,
    ct_housingunits_renter_occupied,
    ct_housingunits_renter_occupied_no_vehicle,
    ct_snap_retailers,
    ct_food_outposts,
    name,
    code,
    division_id,
    region_id
)
WITH snap_agg AS (
  SELECT
    st.id,
    COUNT(DISTINCT snap.id) AS ct_snap_retailers
  FROM states st
  LEFT JOIN snap_retailers snap
    ON st.geometry && snap.geometry
   AND ST_Contains(st.geometry, snap.geometry)
  GROUP BY st.id
),
outpost_agg AS (
  SELECT
    st.id,
    COUNT(DISTINCT o.id) AS ct_food_outposts
  FROM states st
  LEFT JOIN efos_deduped o
    ON st.geometry && o.geometry
   AND ST_Contains(st.geometry, o.geometry)
  GROUP BY st.id
),
agg AS (
  SELECT st.id,
         COALESCE(s.ct_snap_retailers, 0) AS ct_snap_retailers,
         COALESCE(o.ct_food_outposts, 0) AS ct_food_outposts
  FROM states st
  LEFT JOIN snap_agg s USING (id)
  LEFT JOIN outpost_agg o USING (id)
)
SELECT
    st.id,
    survey."ASN1E001"::integer as ct_population,
    survey."ASRAE001"::integer as ct_households,
    survey."ASRAE002"::integer as ct_households_any_food_assistance,
    survey."ASSKE002"::integer as ct_households_snap_food_assistance,
    survey."ASSKE003"::integer as ct_households_snap_food_assistance_disability,
    survey."ASUTE002"::integer as ct_housingunits,
    survey."ASUTE002"::integer as ct_housingunits_owner_occupied,
    survey."ASUTE003"::integer as ct_housingunits_owner_occupied_no_vehicle,
    survey."ASUTE009"::integer as ct_housingunits_renter_occupied,
    survey."ASUTE010"::integer as ct_housingunits_renter_occupied_no_vehicle,
    agg.ct_snap_retailers as ct_snap_retailers,
    agg.ct_food_outposts as ct_food_outposts,
    st.name,
    st.code,
    st.division_id,
    st.region_id
from states st
INNER JOIN "acs_2023_state_raw" survey
    ON st.id = survey."TL_GEO_ID"
LEFT JOIN agg
    ON st.id = agg.id

 * postgresql://healthgps:***@db/healthgps
Done.
Done.
52 rows affected.


[]

In [327]:
%%sql
-- Extend createa analytics view for state metrics

CREATE VIEW state_analytics_view
AS
SELECT
    sm.*,
    s.geometry
FROM state_metrics_expanded sm
JOIN states s
    on sm.id = s.id;

 * postgresql://healthgps:***@db/healthgps
Done.


[]

In [329]:
%%sql
-- Extend state table with metrics from Feeding America

ALTER TABLE state_metrics
    ADD COLUMN IF NOT EXISTS ct_cpp_fss_surveys NUMERIC,
    ADD COLUMN IF NOT EXISTS ct_cpp_fss_households NUMERIC,
    ADD COLUMN IF NOT EXISTS ct_cpp_fss_hesc3 NUMERIC,
    ADD COLUMN IF NOT EXISTS ct_cpp_fss_hesc3_response_yes NUMERIC,
    ADD COLUMN IF NOT EXISTS ct_cpp_fss_hesc3_response_no NUMERIC,
    ADD COLUMN IF NOT EXISTS ct_cpp_fss_hesc3a NUMERIC,
    ADD COLUMN IF NOT EXISTS ct_cpp_fss_hesc3a_response_yes NUMERIC,
    ADD COLUMN IF NOT EXISTS ct_cpp_fss_hesc3a_response_no NUMERIC,
    ADD COLUMN IF NOT EXISTS norm_hesc3_response_yes DECIMAL,
    ADD COLUMN IF NOT EXISTS norm_hesc3a_response_yes DECIMAL;


UPDATE state_metrics c
SET

    ct_cpp_fss_surveys = x.ct_cpp_fss_surveys,
    ct_cpp_fss_households = x.ct_cpp_fss_households,
    ct_cpp_fss_hesc3 = x.ct_cpp_fss_hesc3,
    ct_cpp_fss_hesc3_response_yes = x.ct_cpp_fss_hesc3_response_yes,
    ct_cpp_fss_hesc3_response_no = x.ct_cpp_fss_hesc3_response_no,
    ct_cpp_fss_hesc3a = x.ct_cpp_fss_hesc3a,
    ct_cpp_fss_hesc3a_response_yes = x.ct_cpp_fss_hesc3a_response_yes,
    ct_cpp_fss_hesc3a_response_no = x.ct_cpp_fss_hesc3a_response_no,
    norm_hesc3_response_yes = x.norm_hesc3_response_yes,
    norm_hesc3a_response_yes = x.norm_hesc3a_response_yes

FROM (
    with hh_survey AS (
        select
            ("HRHHID"::text || '-' || "HRHHID2"::text) as household_id,
            ("HRMONTH"::text || '-' || "HRYEAR4"::text) as survey_cycle,
            COALESCE(
                MAX("GESTFIPS") FILTER (WHERE "GESTFIPS" IS NOT NULL),
                MAX("GCFIP")    FILTER (WHERE "GCFIP"    IS NOT NULL)
            ) AS state_fips,
            MAX("HESC3") filter (where "HESC3" > 0) as hesc3_response,
            MAX("HESC3A") filter (where "HESC3A" > 0) as hesc3a_response,
            MAX("HHSUPWGT" / 10000) AS weight
        from census_cps_fss_raw
        WHERE "HHSUPWGT" > 0
        AND ("HESC3" > 0 OR "HESC3A" > 0)
        group by 1,2
    )
    SELECT
        state_fips,
        COUNT(1) AS ct_cpp_fss_surveys,
        COUNT(distinct household_id) AS ct_cpp_fss_households,
        COUNT(1) FILTER (WHERE hesc3_response IS NOT NULL) AS ct_cpp_fss_hesc3,
        COUNT(1) FILTER (WHERE hesc3_response = 1) AS ct_cpp_fss_hesc3_response_yes,
        COUNT(1) FILTER (WHERE hesc3_response = 2) AS ct_cpp_fss_hesc3_response_no,
        COUNT(1) FILTER (WHERE hesc3a_response IS NOT NULL) AS ct_cpp_fss_hesc3a,
        COUNT(1) FILTER (WHERE hesc3a_response = 1) AS ct_cpp_fss_hesc3a_response_yes,
        COUNT(1) FILTER (WHERE hesc3a_response = 2) AS ct_cpp_fss_hesc3a_response_no,
        SUM(
            weight *
            CASE WHEN hesc3_response = 1 THEN 1 ELSE 0 END
        ) / 5.0 AS norm_hesc3_response_yes,
        SUM(
            weight *
            CASE WHEN hesc3a_response = 1 THEN 1 ELSE 0 END
        ) / 5.0 AS norm_hesc3a_response_yes
    FROM hh_survey
    group by 1
) x
WHERE c.id::text = x.state_fips::text;

 * postgresql://healthgps:***@db/healthgps
Done.
44 rows affected.


[]

In [142]:
%%sql
-- Extend county_agg table with metrics from Feeding America
ALTER TABLE counties_metrics
    ADD COLUMN IF NOT EXISTS fa_high_income_threshold DECIMAL,
    ADD COLUMN IF NOT EXISTS fa_low_income_threshold DECIMAL,
    ADD COLUMN IF NOT EXISTS fa_high_income_pct DECIMAL,
    ADD COLUMN IF NOT EXISTS fa_between_income_pct DECIMAL,
    ADD COLUMN IF NOT EXISTS fa_low_income_pct DECIMAL,
    ADD COLUMN IF NOT EXISTS fa_low_income_threshold_programs TEXT,
    ADD COLUMN IF NOT EXISTS fa_high_income_threshold_programs TEXT,
    ADD COLUMN IF NOT EXISTS fa_pop NUMERIC,
    ADD COLUMN IF NOT EXISTS fa_fi_rate DECIMAL,
    ADD COLUMN IF NOT EXISTS fa_fi_pop NUMERIC,
    ADD COLUMN IF NOT EXISTS fa_child_fi_rate DECIMAL,
    ADD COLUMN IF NOT EXISTS fa_child_fi_pop NUMERIC,
    ADD COLUMN IF NOT EXISTS fa_child_fi_below_pct DECIMAL,
    ADD COLUMN IF NOT EXISTS fa_child_fi_above_pct DECIMAL,
    ADD COLUMN IF NOT EXISTS fa_cpm DECIMAL,
    ADD COLUMN IF NOT EXISTS fa_annual_gap_funding_needed NUMERIC,
    ADD COLUMN IF NOT EXISTS fa_associated_bank_ids TEXT[];


UPDATE counties_metrics c
SET
    fa_high_income_threshold = (x.finding->>'HI_TH')::DECIMAL,
    fa_low_income_threshold = (x.finding->>'LOW_TH')::DECIMAL,
    fa_high_income_pct = (x.finding->>'HI_PCT')::DECIMAL,
    fa_between_income_pct = (x.finding->>'BTWN_PCT')::DECIMAL,
    fa_low_income_pct = (x.finding->>'LOW_PCT')::DECIMAL,
    fa_low_income_threshold_programs = x.finding->>'LOW_TH_PROGS',
    fa_high_income_threshold_programs = x.finding->>'HI_TH_PROGS',
    fa_pop = (x.finding->>'COUNTY_POP')::NUMERIC,
    fa_fi_rate = (x.finding->>'COUNTY_FI_RATE')::DECIMAL,
    fa_fi_pop = (x.finding->>'COUNTY_POP_FI')::NUMERIC,
    fa_child_fi_rate = (x.finding->>'CHILD_FI_PCT')::DECIMAL,
    fa_child_fi_pop = (x.finding->>'CHILD_FI_COUNT')::NUMERIC,
    fa_child_fi_below_pct = (x.finding->>'CHILD_FI_BELOW_PCT')::DECIMAL,
    fa_child_fi_above_pct = (x.finding->>'CHILD_FI_ABOVE_PCT')::DECIMAL,
    fa_cpm = (x.finding->>'COST_PER_MEAL')::DECIMAL,
    fa_annual_gap_funding_needed = (x.finding->>'WT_ANNUAL_DOLLARS')::NUMERIC,
    fa_associated_bank_ids = x.fa_associated_bank_ids
FROM (
    WITH parsed_capacity AS (
        SELECT
            "EntityID" as id,
                REPLACE(
                    REPLACE(
                        REPLACE(
                            REPLACE(
                                REPLACE(
                                    REPLACE(
                                        REPLACE("ListFipsCounty", ' ', ''),
                                        ''',','",'
                                    ),
                                    ''':','":'
                                ),
                                ':''',
                                ':"'
                            ),
                            ',''',
                            ',"'
                        ),
                        '{''',
                        '{"'
                    ),
                    '''}',
                    '"}'
                ) as cleaned_json_text
        FROM feeding_america_foodbanks_raw
    ),
    county_arrays as (
        select
            id,
            cleaned_json_text::jsonb -> 'LocalFindings' AS local_findings
        from parsed_capacity
    )
    select
        elem->>'FipsCode' as county_id,
        elem AS finding,
        array_agg(id) as fa_associated_bank_ids
    from
    county_arrays
    cross join lateral (
        select jsonb_array_elements(
            case
                when local_findings is null then '[]'::jsonb
                when jsonb_typeof(local_findings) = 'array' then local_findings
                else jsonb_build_array(local_findings)
            end
        ) as elem
    )
    group by 1,2
) x
WHERE c.id = x.county_id;

 * postgresql://healthgps:***@db/healthgps
Done.
3144 rows affected.


[]

In [100]:
%%sql
-- Add in metrics to filter out on poverty lines
ALTER TABLE counties_metrics
    -- DROP COLUMN filter_step_metric,
    -- DROP COLUMN filter_step_bucket,
    ADD COLUMN IF NOT EXISTS  filter_step_metric DECIMAL,
    ADD COLUMN IF NOT EXISTS filter_step_bucket INTEGER;

UPDATE counties_metrics c
SET
    filter_step_metric = x.z_metric,
    filter_step_bucket = x.z_metric_bucket
FROM (
    WITH init AS (
        SELECT
            id,
            name,
            state_id,
            state_code,
            ct_income_below_poverty,
            ct_poverty_determined_population
        FROM counties_metrics
        WHERE state_code != 'PR'
    ),
    metrics AS (
        SELECT
            id,
            name,
            state_id,
            state_code,
            1.0 * ct_income_below_poverty / ct_poverty_determined_population as metric
        FROM init
    ),
    stats AS (
        SELECT
            avg(metric) as avg_metric,
            stddev_pop(metric) as sd_metric
        FROM metrics
    )
    SELECT
        m.id as id,
        m.name,
        m.state_code,
        m.metric as z_metric,
        CASE
            WHEN abs((m.metric - s.avg_metric) / s.sd_metric) BETWEEN -1 AND 1 THEN 1
            WHEN ((m.metric - s.avg_metric) / s.sd_metric) BETWEEN -2 AND -1 THEN -2
            WHEN ((m.metric - s.avg_metric) / s.sd_metric) BETWEEN 1 AND 2 THEN 2
            WHEN ((m.metric - s.avg_metric) / s.sd_metric) BETWEEN -3 AND -2 THEN -3
            WHEN ((m.metric - s.avg_metric) / s.sd_metric) BETWEEN 2 AND 3 THEN 3
            WHEN ((m.metric - s.avg_metric) / s.sd_metric) < -3 THEN -4
            WHEN ((m.metric - s.avg_metric) / s.sd_metric) > 3 THEN 4
            ELSE 1000
        END AS z_metric_bucket
    FROM metrics m
    CROSS JOIN stats s
) x
WHERE c.id = x.id;

Running query in 'postgresql://healthgps:***@db/healthgps'

3144 rows affected.

++
||
++
++

In [ ]:
# INGEST census_cps_fss_raw VIA PANDAS

In [332]:
%%sql
DROP VIEW counties_analytics_view;

CREATE VIEW
  counties_analytics_view AS
WITH cav AS (
    SELECT
      a.*,
      case
        when (ct_food_outposts + ct_snap_retailers) = 0 then null
        else 100. * ct_snap_retailers / (ct_food_outposts + ct_snap_retailers)
      end AS pct_snap_retailers,
      case
        when (ct_food_outposts + ct_snap_retailers) = 0 then null
        else (ct_income_below_poverty / (1.0 * ct_poverty_determined_population)) * (1.0 * ct_snap_retailers / (ct_food_outposts + ct_snap_retailers))
      end AS pct_metric,
      c.geometry
    FROM
      counties_metrics a
    LEFT JOIN counties c
        ON a.id = c.id
),
norms AS (
    SELECT max(pct_metric) - min(pct_metric) as norm
    FROM cav
)
SELECT
    *,
    100.0 * pct_metric / norm as pct_metric_norm
FROM cav
CROSS JOIN norms

 * postgresql://healthgps:***@db/healthgps
Done.
Done.


[]

In [247]:
df_request = %sql select * from counties_analytics_view

 * postgresql://healthgps:***@db/healthgps
3222 rows affected.


In [286]:
df = df_request.DataFrame()
df = df[df["state_code"] != "PR"]
df["pct_snap_retailers"] = df["pct_snap_retailers"].astype(float)
def categorize(ct):
    if ct < 5:
        return "exclude"
    elif ct >= 5:
        return "include"
    else:
        return "exclude"

df["cat"] = df["ct_snap_retailers"].apply(categorize)

avg = df['pct_snap_retailers'].mean()
std = df['pct_snap_retailers'].std()

In [287]:
df[["id","state_code", "cat"]].groupby(["state_code", "cat"]).count()

id
state_code cat        
AK         exclude   9
           include  21
AL         include  67
AR         exclude   1
           include  74
...                 ..
WI         exclude   1
           include  71
WV         include  55
WY         exclude   3
           include  20

[85 rows x 1 columns]

In [284]:
df["cat"]

0        normal
1        normal
2        normal
3           red
4       exclude
         ...   
3217        red
3218        red
3219        red
3220        red
3221        red
Name: cat, Length: 3144, dtype: str